# Manifold walking, α sweep, and the multi-layer steering catastrophe

Part 4 of the *Manifold Features* series, main positive result. Three experiments fused into one notebook, in order of increasing aggressiveness of intervention:

- **Section A — 50-step centroid walk** along the path `Mon → Tue → Wed → Thu → Fri` at `L=18` (and at `L=24` for a sanity comparison). At each interpolated point we replace the last-token residual with the path centroid and read out `P(day)`. The straight chord through 4096-d residual space teleports through `Sunday`; the manifold path produces a clean ordered hand-off through the weekdays.
- **Section B — single-layer α sweep at L=18.** Goodfire-style Fourier-rotation steering with `α ∈ {1, 2, 3, 5, 7, 10, 15, 20}` on addition, weekdays, months. The published `α = 10` over-shoots cross-task; the optimum is `α = 5` (65% weekdays, 79% months).
- **Section C — multi-layer steering catastrophe.** Stack Goodfire's `α = 10` rotation across {1, 2, 3, 5} consecutive layers around L=18. Mean argmax hit rate on weekdays falls from ~44% (1 layer) to ~6% (5 layers); months from ~60% to ~0%. Addition (probes' native task) is only mildly affected — the cross-task collapse is the catastrophe.

**Source model**: `meta-llama/Llama-3.1-8B` (gated; needs `HF_TOKEN`).

**Outputs**: `./out/manifold_walking/{walk_trajectories.json, alpha_sweep.json, multilayer_steering.json}` + three figures.

**Runs on**: a single A100 (40 GB) in ~30 minutes total. With `RECOMPUTE = False` each section replots from the bundled `../data/*.json` in seconds.

Order of execution. Section A is self-contained: it reads `../goodfire_replication/out/centroids_L18.npz` if present, otherwise it captures fresh centroids. Sections B and C train fresh Ridge probes on addition residuals at L=18-22 — they do not need the centroids cache.


In [ ]:
# Cell 1 — setup
import os, sys, json, time, subprocess
from pathlib import Path

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import numpy as np
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from torch.nn.functional import softmax
    from sklearn.linear_model import Ridge
    import matplotlib.pyplot as plt
except ImportError:
    _pip('torch', 'transformers', 'numpy', 'scikit-learn', 'matplotlib', 'huggingface_hub')
    import numpy as np
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from torch.nn.functional import softmax
    from sklearn.linear_model import Ridge
    import matplotlib.pyplot as plt

# HF token resolution (Colab userdata, then env)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    raise RuntimeError('Llama 3.1 8B is gated. Set HF_TOKEN as an env var or Colab secret.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'meta-llama/Llama-3.1-8B'
MAX_NEW_TOKENS = 512

BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.5, 'axes.labelcolor': FG,
    'axes.titlecolor': FG, 'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.6,
})


RECOMPUTE = True

OUT_DIR = Path('./out/manifold_walking').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

PERIODS = [2, 5, 10, 20, 50, 100]
STEER_LAYERS = [18, 19, 20, 21, 22]
TARGET_LAYER = 18
RIDGE_ALPHA = 100.0
SEED = 42
N_ADD = 2000
N_SWEEP_POINTS = 50
WALK_LAYERS = [18, 24]

# Where to look for centroids cached by goodfire_replication.ipynb
CENTROIDS_CACHE = Path('../goodfire_replication/out/centroids_L18.npz')

DATA_FALLBACK_WALK   = Path('../data/manifold_walk_trajectories.json')
DATA_FALLBACK_ALPHA  = Path('../data/alpha_sweep_results.json')
DATA_FALLBACK_MULTI  = Path('../data/multilayer_steering_results.json')


In [ ]:
# Cell 2 — load Llama (only if RECOMPUTE)
if RECOMPUTE:
    # Load Llama-3.1-8B (gated; needs HF_TOKEN)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tok.padding_side = 'left'
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    
    t0 = time.time()
    MODEL = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        attn_implementation='sdpa', device_map='auto', token=HF_TOKEN,
    ).eval()
    for p in MODEL.parameters():
        p.requires_grad_(False)
    print(f'Loaded {MODEL_NAME} in {time.time()-t0:.1f}s on {DEVICE}')
    

else:
    print('RECOMPUTE = False; will replot from bundled JSON.')


## Section A — 50-step manifold walk vs linear interpolation

50-step centroid walk along `Mon → Tue → Wed → Thu → Fri` at `L=18` and `L=24`. At each of 50 interpolated points the last-token residual is replaced by the path centroid via a forward hook, and `P(day)` is measured across a held-out subset of weekday-correct prompts.


In [ ]:
# Cell 3A — build per-day centroids (load from cache if available)
DAYS = ['Saturday','Sunday','Monday','Tuesday','Wednesday','Thursday','Friday']
DAY_TO_IDX = {d: i for i, d in enumerate(DAYS)}
OFFSETS_W = ['one','two','three','four','five','six','seven','eight','nine','ten',
             'eleven','twelve','thirteen','fourteen']
PATH_DAYS = ['Monday','Tuesday','Wednesday','Thursday','Friday']

def first_tid_of(name, needs_space=True):
    text = (' ' + name) if needs_space else name
    return tok.encode(text, add_special_tokens=False)[0]

if RECOMPUTE:
    weekday_prompts = []
    for d in DAYS:
        for i, o in enumerate(OFFSETS_W):
            target = DAYS[(DAY_TO_IDX[d] + i + 1) % 7]
            weekday_prompts.append({'prompt': f'Q: What day is {o} days after {d}?\nA:',
                                    'expected': target})
    DAY_TID = {d: first_tid_of(d, needs_space=True) for d in DAYS}

    @torch.no_grad()
    def capture_per_prompt(prompts_list, layers, batch_size=16):
        per_layer = {L: [] for L in layers}
        argmaxes = []
        for i in range(0, len(prompts_list), batch_size):
            batch = prompts_list[i:i + batch_size]
            ids = tok([p['prompt'] for p in batch], return_tensors='pt', padding=True).to(DEVICE)
            captured = {L: None for L in layers}
            handles = []
            def make_hk(L):
                def hk(m, inp, out):
                    h = out[0] if isinstance(out, tuple) else out
                    captured[L] = h[:, -1, :].detach().float().cpu()
                return hk
            for L in layers:
                handles.append(MODEL.model.layers[L].register_forward_hook(make_hk(L)))
            try: logits = MODEL(**ids).logits
            finally:
                for h in handles: h.remove()
            for L in layers: per_layer[L].append(captured[L])
            argmaxes.extend(logits[:, -1].argmax(-1).cpu().tolist())
        return {L: torch.cat(per_layer[L], dim=0).numpy() for L in layers}, np.array(argmaxes)

    if CENTROIDS_CACHE.exists():
        cache = np.load(CENTROIDS_CACHE, allow_pickle=True)
        cached_days = list(cache['days'])
        M = {WALK_LAYERS[0]: {d: cache[f'M_{d}'] for d in cached_days if f'M_{d}' in cache.files}}
        print(f'Loaded cached centroids for L={WALK_LAYERS[0]} from {CENTROIDS_CACHE}')
        # Still need L=24 centroids → capture both layers' activations together
        acts, base_argmax = capture_per_prompt(weekday_prompts, WALK_LAYERS)
        is_correct = np.array([p['expected'].lower() in tok.decode(base_argmax[k]).lower()
                               for k, p in enumerate(weekday_prompts)])
        for L in WALK_LAYERS:
            if L in M: continue
            M[L] = {}
            for d in DAYS:
                mask = np.array([(p['expected'] == d and ok)
                                 for p, ok in zip(weekday_prompts, is_correct)])
                if mask.sum() > 0:
                    M[L][d] = acts[L][mask].mean(axis=0)
    else:
        print(f'No centroids cache at {CENTROIDS_CACHE}; capturing fresh activations at L={WALK_LAYERS}...')
        acts, base_argmax = capture_per_prompt(weekday_prompts, WALK_LAYERS)
        is_correct = np.array([p['expected'].lower() in tok.decode(base_argmax[k]).lower()
                               for k, p in enumerate(weekday_prompts)])
        M = {}
        for L in WALK_LAYERS:
            M[L] = {}
            for d in DAYS:
                mask = np.array([(p['expected'] == d and ok)
                                 for p, ok in zip(weekday_prompts, is_correct)])
                if mask.sum() > 0:
                    M[L][d] = acts[L][mask].mean(axis=0)
                    print(f'  L={L} M[{d}]: n={mask.sum()}, ||M||={np.linalg.norm(M[L][d]):.2f}')

    # Pick query subset for trajectory evaluation
    correct_prompts = [p for p, ok in zip(weekday_prompts, is_correct) if ok]
    rng_q = np.random.default_rng(42)
    query_subset = list(rng_q.choice(correct_prompts,
                                      size=min(10, len(correct_prompts)), replace=False))
    print(f'Query subset size: {len(query_subset)}')


In [ ]:
# Cell 4A — replace-last-token hook + path builders, then 50-step sweep at L=18, L=24
if RECOMPUTE:
    class ReplaceLastTokenAt:
        def __init__(self, layer_idx, new_vec):
            self.layer_idx = layer_idx; self.new_vec = new_vec; self._h = None
        def __enter__(self):
            new_t = torch.from_numpy(self.new_vec).to(DEVICE, dtype=torch.bfloat16)
            def hook(m, inp, out):
                h = out[0] if isinstance(out, tuple) else out
                h[:, -1, :] = new_t
                return (h,) + out[1:] if isinstance(out, tuple) else h
            self._h = MODEL.model.layers[self.layer_idx].register_forward_hook(hook)
            return self
        def __exit__(self, *a): self._h.remove()

    @torch.no_grad()
    def forward_with_replace(prompt, layer, new_vec):
        ids = tok(prompt, return_tensors='pt').input_ids.to(DEVICE)
        with ReplaceLastTokenAt(layer, new_vec):
            out = MODEL(ids)
        return softmax(out.logits[0, -1].float(), dim=-1).cpu().numpy()

    def build_manifold_path(centroids, days_path, n_pts):
        path_len = len(days_path) - 1
        out = []
        for i in range(n_pts):
            t = i / (n_pts - 1)
            seg_t = t * path_len
            seg_idx = min(int(seg_t), path_len - 1)
            local_t = seg_t - seg_idx
            a = centroids[days_path[seg_idx]]
            b = centroids[days_path[seg_idx + 1]]
            out.append((1 - local_t) * a + local_t * b)
        return np.stack(out, axis=0)

    def build_linear_path(centroids, start_day, end_day, n_pts):
        a = centroids[start_day]; b = centroids[end_day]
        return np.stack([(1 - i/(n_pts-1)) * a + (i/(n_pts-1)) * b
                         for i in range(n_pts)], axis=0)

    TRAJ_RESULTS = {}
    for L in WALK_LAYERS:
        if not all(d in M[L] for d in PATH_DAYS):
            print(f'  L={L}: missing centroids for some path days; skipping')
            continue
        manifold_path = build_manifold_path(M[L], PATH_DAYS, N_SWEEP_POINTS)
        linear_path   = build_linear_path(M[L], 'Monday', 'Friday', N_SWEEP_POINTS)
        for method_name, path_pts in [('manifold', manifold_path), ('linear', linear_path)]:
            t0 = time.time()
            P_traj = np.zeros((N_SWEEP_POINTS, len(DAYS)))
            for ti, pt in enumerate(path_pts):
                probs_at_t = np.zeros((len(query_subset), len(DAYS)))
                for qi, p in enumerate(query_subset):
                    probs = forward_with_replace(p['prompt'], L, pt)
                    for di, d in enumerate(DAYS):
                        probs_at_t[qi, di] = probs[DAY_TID[d]]
                P_traj[ti] = probs_at_t.mean(axis=0)
            TRAJ_RESULTS[f'L{L}_{method_name}'] = P_traj.tolist()
            print(f'  L={L} {method_name}: {N_SWEEP_POINTS} pts in {time.time()-t0:.0f}s')

    with open(OUT_DIR / 'walk_trajectories.json', 'w') as f:
        json.dump({'days': DAYS, 'path_days': PATH_DAYS, 'n_sweep': N_SWEEP_POINTS,
                   'query_subset_size': len(query_subset),
                   'trajectories': TRAJ_RESULTS}, f, indent=2)
    print(f'Saved walk_trajectories.json')
else:
    with open(DATA_FALLBACK_WALK) as f:
        walk_blob = json.load(f)
    TRAJ_RESULTS = walk_blob['trajectories']
    DAYS = walk_blob['days']
    PATH_DAYS = walk_blob['path_days']
    N_SWEEP_POINTS = walk_blob['n_sweep']
    print(f'Loaded walk trajectories from {DATA_FALLBACK_WALK}')


In [ ]:
# Cell 5A — Figure: 50-step P(day) trajectories at L=18 and L=24, manifold vs linear
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
day_colors = {'Saturday': '#888', 'Sunday': '#aa44aa', 'Monday': '#cc4444',
              'Tuesday': '#dd8844', 'Wednesday': '#cccc44', 'Thursday': '#44aa44',
              'Friday': '#4477cc'}
configs = [('L=18', 18, 'manifold'), ('L=18', 18, 'linear'),
           ('L=24', 24, 'manifold'), ('L=24', 24, 'linear')]
for ax, (lbl, L, method) in zip(axes.flat, configs):
    key = f'L{L}_{method}'
    if key not in TRAJ_RESULTS:
        ax.set_title(f'{lbl}, {method}: not computed'); continue
    P_traj = np.array(TRAJ_RESULTS[key])
    ts = np.linspace(0, 1, N_SWEEP_POINTS)
    for di, d in enumerate(DAYS):
        ax.plot(ts, P_traj[:, di], color=day_colors[d], label=d,
                linewidth=1.6 if d in PATH_DAYS else 0.8,
                alpha=1.0 if d in PATH_DAYS else 0.4)
    if method == 'manifold':
        for i, d in enumerate(PATH_DAYS):
            ax.axvline(i / (len(PATH_DAYS) - 1), ls=':', color=day_colors[d], alpha=0.4)
    ax.set_xlabel('Sweep position t')
    ax.set_ylabel('mean P(day)')
    ax.set_title(f'{lbl}, {method} walk Mon→Fri (50 points)')
    ax.legend(fontsize=7, ncol=2, loc='upper right')
    ax.set_ylim(0, max(0.5, P_traj.max() * 1.1))
plt.tight_layout()
plt.savefig(OUT_DIR / 'walk_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {OUT_DIR/"walk_trajectories.png"}')


## Section B — α sweep at single layer L=18

Goodfire-style Fourier-rotation steering with α ∈ {1, 2, 3, 5, 7, 10, 15, 20}. Same probe-trained directions as `goodfire_replication.ipynb`; here we re-train them locally so this notebook is self-contained.


In [ ]:
# Cell 6B — capture residuals + train fresh Ridge probes at L=18..22
if RECOMPUTE:
    rng_d = np.random.default_rng(SEED)
    add_pairs = [(int(rng_d.integers(0, 100)), int(rng_d.integers(0, 100))) for _ in range(N_ADD)]
    add_prompts = [f'{a}+{b}=' for a, b in add_pairs]
    add_sums = np.array([a + b for a, b in add_pairs])

    @torch.no_grad()
    def capture_layers_batch(prompts, layer_ids, batch_size=16):
        per_layer = {L: [] for L in layer_ids}
        for i in range(0, len(prompts), batch_size):
            ids = tok(prompts[i:i + batch_size], return_tensors='pt', padding=True).to(DEVICE)
            captured = {L: None for L in layer_ids}
            handles = []
            def make_hk(L):
                def hk(m, inp, out):
                    h = out[0] if isinstance(out, tuple) else out
                    captured[L] = h[:, -1, :].detach().float().cpu()
                return hk
            for L in layer_ids:
                handles.append(MODEL.model.layers[L].register_forward_hook(make_hk(L)))
            try: MODEL(**ids)
            finally:
                for h in handles: h.remove()
            for L in layer_ids: per_layer[L].append(captured[L])
        return {L: torch.cat(per_layer[L], dim=0).numpy() for L in layer_ids}

    print(f'Capturing {N_ADD} addition residuals at L={STEER_LAYERS}...')
    R_per_L = capture_layers_batch(add_prompts, STEER_LAYERS)

    def train_ridge_probes(R, sums, periods=PERIODS, alpha=RIDGE_ALPHA, test_frac=0.2, seed=SEED):
        n = len(sums); rng = np.random.RandomState(seed)
        idx = rng.permutation(n); n_tr = int(n * (1 - test_frac))
        tr, te = idx[:n_tr], idx[n_tr:]
        out = {}
        for T in periods:
            sin_t = np.sin(2 * np.pi * sums / T); cos_t = np.cos(2 * np.pi * sums / T)
            sp = Ridge(alpha=alpha).fit(R[tr], sin_t[tr])
            cp = Ridge(alpha=alpha).fit(R[tr], cos_t[tr])
            out[T] = {'sin_w': sp.coef_.astype(np.float32), 'sin_b': float(sp.intercept_),
                      'cos_w': cp.coef_.astype(np.float32), 'cos_b': float(cp.intercept_)}
        return out

    PROBES_PER_L = {L: train_ridge_probes(R_per_L[L], add_sums) for L in STEER_LAYERS}
    print('Probes ready at L=18-22')


In [ ]:
# Cell 7BC — shared task definitions (addition / weekdays / months) + steering helpers
if RECOMPUTE:
    MONTHS = ['January','February','March','April','May','June',
              'July','August','September','October','November','December']
    OFFSETS_M = OFFSETS_W + ['fifteen','sixteen','seventeen','eighteen','nineteen','twenty',
                              'twenty-one','twenty-two','twenty-three','twenty-four']

    TASKS = {
        'addition': {
            'items': [str(s) for s in range(0, 24)],
            'steer_sums': list(range(0, 24)),
            'needs_space': False,
            'sum_to_item': lambda s: str(s),
            'check': lambda e, p: p.strip() == e,
            'make_prompts': lambda: [
                {'prompt': f'{a}+{b}=', 'expected': str(a + b), 'raw_sum': a + b}
                for a in range(1, 12) for b in range(1, 12)],
        },
        'weekdays': {
            'items': DAYS,
            'steer_sums': list(range(0, 7)),
            'needs_space': True,
            'sum_to_item': lambda s: DAYS[s % 7],
            'check': lambda e, p: e.lower() in p.lower(),
            'make_prompts': lambda: [
                {'prompt': f'Q: What day is {o} days after {d}?\nA:',
                 'expected': DAYS[(DAY_TO_IDX[d] + (i + 1)) % 7],
                 'raw_sum': DAY_TO_IDX[d] + (i + 1)}
                for d in DAYS for i, o in enumerate(OFFSETS_W)],
        },
        'months': {
            'items': MONTHS,
            'steer_sums': list(range(1, 13)),
            'needs_space': True,
            'sum_to_item': lambda s: MONTHS[(s - 1) % 12],
            'check': lambda e, p: e.lower() in p.lower(),
            'make_prompts': lambda: [
                {'prompt': f'Q: What month is {o} months after {m}?\nA:',
                 'expected': MONTHS[((mi + 1) + (oi + 1) - 1) % 12],
                 'raw_sum': (mi + 1) + (oi + 1)}
                for mi, m in enumerate(MONTHS) for oi, o in enumerate(OFFSETS_M)],
        },
    }
    for tname, t in TASKS.items():
        t['item_tid'] = {it: first_tid_of(it, t['needs_space']) for it in t['items']}
        t['prompts'] = t['make_prompts']()
        print(f'  {tname}: {len(t["prompts"])} prompts')

    def compute_mod(r_query, probes_at_L, target_n, alpha, periods=PERIODS):
        patched = r_query.copy()
        orig_r = {}
        for T in periods:
            pw = probes_at_L[T]
            s = pw['sin_w'] @ patched + pw['sin_b']
            c = pw['cos_w'] @ patched + pw['cos_b']
            orig_r[T] = float(np.sqrt(s * s + c * c))
        for T in periods:
            pw = probes_at_L[T]
            theta = 2 * np.pi * target_n / T
            r_t = orig_r[T] * alpha
            target_s = r_t * np.sin(theta); target_c = r_t * np.cos(theta)
            w_sin_n = float(np.linalg.norm(pw['sin_w']))
            w_cos_n = float(np.linalg.norm(pw['cos_w']))
            d_sin = pw['sin_w'] / (w_sin_n + 1e-9)
            d_cos = pw['cos_w'] / (w_cos_n + 1e-9)
            cur_s = pw['sin_w'] @ patched + pw['sin_b']
            patched = patched + ((target_s - cur_s) / w_sin_n) * d_sin
            cur_c = pw['cos_w'] @ patched + pw['cos_b']
            patched = patched + ((target_c - cur_c) / w_cos_n) * d_cos
        return patched - r_query

    class MultiLayerSteer:
        def __init__(self, layer_to_mod):
            self.layer_to_mod = layer_to_mod; self._handles = []
        def __enter__(self):
            for L, mod_np in self.layer_to_mod.items():
                mod_t = torch.from_numpy(mod_np).to(DEVICE, dtype=torch.bfloat16)
                def make_hook(mod):
                    def hook(m, inp, out):
                        h = out[0] if isinstance(out, tuple) else out
                        h[:, -1, :] = h[:, -1, :] + mod
                        return (h,) + out[1:] if isinstance(out, tuple) else h
                    return hook
                self._handles.append(MODEL.model.layers[L].register_forward_hook(make_hook(mod_t)))
            return self
        def __exit__(self, *a):
            for h in self._handles: h.remove()

    @torch.no_grad()
    def steer_multi(prompt, target_n, layers_used, alpha_per_L):
        ids = tok(prompt, return_tensors='pt').input_ids.to(DEVICE)
        captured = {L: None for L in layers_used}
        handles = []
        def make_hk(L):
            def hk(m, inp, out):
                h = out[0] if isinstance(out, tuple) else out
                captured[L] = h[:, -1, :].detach().float().cpu().numpy()[0]
            return hk
        for L in layers_used:
            handles.append(MODEL.model.layers[L].register_forward_hook(make_hk(L)))
        try: MODEL(ids)
        finally:
            for h in handles: h.remove()
        mods = {L: compute_mod(captured[L], PROBES_PER_L[L], target_n, alpha_per_L[L])
                for L in layers_used}
        with MultiLayerSteer(mods):
            return MODEL(ids).logits[0, -1].float()

    @torch.no_grad()
    def measure(task_data, layers_used, alpha_per_L, max_prompts=None):
        prompts = task_data['prompts']
        if max_prompts: prompts = prompts[:max_prompts]
        steer_sums = task_data['steer_sums']; chk = task_data['check']
        item_tid = task_data['item_tid']; sum_to_item = task_data['sum_to_item']
        n_kept = 0
        per_target = {ts: {'hits': 0, 'tot': 0} for ts in steer_sums}
        for p in prompts:
            ids = tok(p['prompt'], return_tensors='pt').input_ids.to(DEVICE)
            base_tok = int(MODEL(ids).logits[0, -1].argmax().item())
            if not chk(p['expected'], tok.decode(base_tok).strip()):
                continue
            n_kept += 1
            for ts in steer_sums:
                target_tid = item_tid[sum_to_item(ts)]
                logits = steer_multi(p['prompt'], ts, layers_used, alpha_per_L)
                per_target[ts]['tot'] += 1
                if int(logits.argmax().item()) == target_tid:
                    per_target[ts]['hits'] += 1
        hit_rates = {ts: per_target[ts]['hits'] / max(1, per_target[ts]['tot']) for ts in steer_sums}
        mean_rate = float(np.mean(list(hit_rates.values())))
        return {'n_kept': n_kept, 'hit_rates': hit_rates, 'mean_rate': mean_rate,
                'layers': layers_used, 'total_budget': float(sum(alpha_per_L.values()))}

    print('Steering helpers ready')


In [ ]:
# Cell 8B — α sweep at L=18 (single layer), three tasks
ALPHA_CONFIGS = [
    ('L18@α1',         [18], {18: 1.0}),
    ('L18@α2',         [18], {18: 2.0}),
    ('L18@α5',         [18], {18: 5.0}),
    ('L18@α10 [GF]',   [18], {18: 10.0}),
    ('L18@α20',        [18], {18: 20.0}),
    # also distributed (used in Section C panel)
    ('L18-19@α5',      [18, 19],            {18: 5.0, 19: 5.0}),
    ('L18-20@α3.33',   [18, 19, 20],        {18: 10/3, 19: 10/3, 20: 10/3}),
    ('L18-22@α2',      [18, 19, 20, 21, 22], {L: 2.0 for L in [18,19,20,21,22]}),
    ('L18-22@α5',      [18, 19, 20, 21, 22], {L: 5.0 for L in [18,19,20,21,22]}),
]

MAX_PER_TASK_AB = {'addition': 30, 'weekdays': 25, 'months': 40}

if RECOMPUTE:
    alpha_results = {tname: {} for tname in TASKS}
    for tname, t in TASKS.items():
        print(f'\n=== {tname} (max {MAX_PER_TASK_AB[tname]} prompts) ===')
        for cname, layers, alpha_per_L in ALPHA_CONFIGS:
            t0 = time.time()
            r = measure(t, layers, alpha_per_L, max_prompts=MAX_PER_TASK_AB[tname])
            print(f'  {cname:>16s}: mean={r["mean_rate"]:.3f}  n_kept={r["n_kept"]}  '
                  f'({time.time()-t0:.0f}s)')
            alpha_results[tname][cname] = r
    with open(OUT_DIR / 'alpha_sweep.json', 'w') as f:
        json.dump({tn: {cn: {**r, 'hit_rates': {str(k): v for k, v in r['hit_rates'].items()}}
                        for cn, r in d.items()} for tn, d in alpha_results.items()}, f, indent=2)
    print(f'\nSaved alpha_sweep.json')
else:
    with open(DATA_FALLBACK_ALPHA) as f:
        alpha_results = json.load(f)
    print(f'Loaded alpha_sweep results from {DATA_FALLBACK_ALPHA}')


In [ ]:
# Cell 9B — Figure: single-layer α sweep + distributed-budget panel
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
task_colors = {'addition': '#cc4444', 'weekdays': '#4477cc', 'months': '#44aa44'}
chance_dict = {'addition': 1/24, 'weekdays': 1/7, 'months': 1/12}

# Left: α sweep at single layer
single_layer_alphas = [('L18@α1', 1.0), ('L18@α2', 2.0), ('L18@α5', 5.0),
                       ('L18@α10 [GF]', 10.0), ('L18@α20', 20.0)]
for tname in ['addition', 'weekdays', 'months']:
    xs = [a for _, a in single_layer_alphas]
    ys = [alpha_results[tname][cn]['mean_rate'] for cn, _ in single_layer_alphas]
    axes[0].plot(xs, ys, '-o', color=task_colors[tname], label=tname, linewidth=2, markersize=8)
    axes[0].axhline(chance_dict[tname], ls=':', color=task_colors[tname], alpha=0.4)
    for x, y in zip(xs, ys):
        axes[0].annotate(f'{y:.2f}', (x, y), textcoords='offset points',
                         xytext=(5, 5), fontsize=8, color=task_colors[tname])
axes[0].set_xscale('log'); axes[0].set_xticks([1, 2, 5, 10, 20])
axes[0].set_xticklabels(['1', '2', '5', '10', '20'])
axes[0].axvline(10, ls='--', color='gray', alpha=0.5, label='Goodfire α=10')
axes[0].set_xlabel('α at L=18 (single layer)')
axes[0].set_ylabel('Mean argmax hit rate')
axes[0].set_title('Single-layer α sweep (peak at α=5)')
axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.05)

# Right: same total budget (10) distributed across {1, 2, 3, 5} layers, plus α=5 × 5 layers
distributed = [
    ('L18@α10 [GF]',  1, 10.0),
    ('L18-19@α5',     2, 10.0),
    ('L18-20@α3.33',  3, 10.0),
    ('L18-22@α2',     5, 10.0),
    ('L18-22@α5',     5, 25.0),
]
for tname in ['addition', 'weekdays', 'months']:
    xs = [n for _, n, _ in distributed]
    ys = [alpha_results[tname][cn]['mean_rate'] for cn, _, _ in distributed]
    axes[1].plot(xs[:4], ys[:4], '-o', color=task_colors[tname],
                 label=f'{tname} (total=10)', linewidth=2, markersize=8)
    axes[1].scatter([xs[4]], [ys[4]], color=task_colors[tname], marker='*', s=200)
    axes[1].axhline(chance_dict[tname], ls=':', color=task_colors[tname], alpha=0.4)
    for x, y in zip(xs, ys):
        axes[1].annotate(f'{y:.2f}', (x, y), textcoords='offset points',
                         xytext=(5, 5), fontsize=8, color=task_colors[tname])
axes[1].set_xlabel('Number of layers (steering distributed across)')
axes[1].set_ylabel('Mean argmax hit rate')
axes[1].set_title('Distributed vs concentrated budget\n(circles: total=10; stars: total=25)')
axes[1].set_xticks([1, 2, 3, 5])
axes[1].legend(fontsize=7); axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_sweep.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {OUT_DIR/"alpha_sweep.png"}')


## Section C — multi-layer steering catastrophe

Goodfire's `α = 10` Fourier rotation, applied simultaneously at {1, 2, 3, 5} consecutive layers starting at L=18. Cross-task accuracy collapses to ~0 at three or more layers; addition (probes' native task) only mildly degrades.


In [ ]:
# Cell 10C — multi-layer steering at fixed per-layer α = 10 (Goodfire default)
MULTI_CONFIGS = [
    ('L18 only', [18]),
    ('L18-19',   [18, 19]),
    ('L18-20',   [18, 19, 20]),
    ('L18-22',   [18, 19, 20, 21, 22]),
]
MAX_PER_TASK_C = {'addition': 60, 'weekdays': 50, 'months': 80}

if RECOMPUTE:
    multi_results = {tname: {} for tname in TASKS}
    for tname, t in TASKS.items():
        print(f'\n=== {tname} ({MAX_PER_TASK_C[tname]} prompts) ===')
        for cname, layers in MULTI_CONFIGS:
            alpha_per_L = {L: 10.0 for L in layers}
            t0 = time.time()
            r = measure(t, layers, alpha_per_L, max_prompts=MAX_PER_TASK_C[tname])
            print(f'  {cname:>10s}: mean={r["mean_rate"]:.3f}  ({time.time()-t0:.0f}s)')
            multi_results[tname][cname] = r
    with open(OUT_DIR / 'multilayer_steering.json', 'w') as f:
        json.dump({tn: {cn: {**r, 'hit_rates': {str(k): v for k, v in r['hit_rates'].items()}}
                        for cn, r in d.items()} for tn, d in multi_results.items()}, f, indent=2)
    print(f'\nSaved multilayer_steering.json')
else:
    with open(DATA_FALLBACK_MULTI) as f:
        multi_results = json.load(f)
    print(f'Loaded multilayer_steering results from {DATA_FALLBACK_MULTI}')


In [ ]:
# Cell 11C — Figure: multi-layer steering catastrophe
configs = [c for c, _ in MULTI_CONFIGS]
n_layers = [len(l) for _, l in MULTI_CONFIGS]
task_colors = {'addition': '#cc4444', 'weekdays': '#4477cc', 'months': '#44aa44'}
chance_dict = {'addition': 1/24, 'weekdays': 1/7, 'months': 1/12}

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(configs)); w = 0.25
for i, tname in enumerate(['addition', 'weekdays', 'months']):
    rates = [multi_results[tname][cn]['mean_rate'] for cn in configs]
    ax.bar(x + (i - 1) * w, rates, w, label=tname, color=task_colors[tname], edgecolor='black')
    ax.axhline(chance_dict[tname], ls=':', color=task_colors[tname], alpha=0.4)
ax.set_xticks(x); ax.set_xticklabels([f'{c}\n({n}L)' for c, n in zip(configs, n_layers)])
ax.set_ylabel('Mean argmax hit rate')
ax.set_title('Multi-layer Fourier rotation at α=10 per layer\nCross-task collapse at 3+ layers')
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUT_DIR / 'multilayer_catastrophe.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {OUT_DIR/"multilayer_catastrophe.png"}')

print('\nSummary:')
print(f'  {"task":>10s}  ' + '  '.join(f'{c:>10s}' for c in configs))
for tname in ['addition', 'weekdays', 'months']:
    print(f'  {tname:>10s}  ' + '  '.join(
        f'{multi_results[tname][c]["mean_rate"]:>10.3f}' for c in configs))


## Expected results

**Section A.**
- `L=18, manifold`: clean ordered handoff. P(Monday) → P(Tuesday) → P(Wednesday) → P(Thursday) → P(Friday) along the sweep.
- `L=18, linear`: the midpoint of the chord between M[Monday] and M[Friday] decodes as **Sunday**, off the path entirely.
- `L=24` (near the head): the manifold path is still smooth, often cleaner than at L=18 — rules out the "downstream averaging" alternative explanation.

**Section B.**
- Single-layer α sweep peaks at **α = 5** for weekdays (~65%) and months (~79%) — both above Goodfire's published default of α = 10. Addition saturates earlier (peak around α = 5-10).
- Distributing the same total budget across more layers (`total = 10` panel) **hurts** cross-task targets while modestly helping addition: the optimum control topology is task-specific.

**Section C.**
- At α=10 per layer, mean weekday hit rate falls from ~44% (1 layer) to ~12% (2 layers) to ~6% (5 layers). Months collapses faster, to ~0%.
- Addition only mildly degrades — because the probes are trained on addition residuals, the perturbation amplitude matches what the model already sees in that direction.

The lesson: aggressive multi-site steering of the same circuit produces out-of-distribution residual activations that downstream layers cannot reconcile. Walking the manifold (Section A) and a gentle single-layer α=5 (Section B) stay inside the support of activations the model has actually seen during training — and they preserve clean cross-task control.
